# 01 — Exploratory Data Analysis

F1 Pit Stop Prediction · Kaggle Playground Series S6E5

**Goal**: Understand the data distribution, class balance, 2023 anomaly, tyre degradation patterns, and race progress dynamics before feature engineering.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data import load_train, load_test, load_external, dedup_external

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

train = load_train()
test = load_test()
ext_raw = load_external()
ext = dedup_external(train, ext_raw)

print(f'Train shape : {train.shape}')
print(f'Test shape  : {test.shape}')
print(f'External (raw): {ext_raw.shape}  |  unique: {ext.shape}')

## 1. Basic overview

In [ ]:
train.head()

In [ ]:
train.dtypes

In [ ]:
train.describe()

In [ ]:
# Missing values
missing = train.isnull().sum()
print('Missing values in train:')
print(missing[missing > 0])

## 2. Class balance

In [ ]:
pos_rate = train['PitNextLap'].mean()
print(f'Overall PitNextLap positive rate: {pos_rate:.4f} ({pos_rate*100:.1f}%)')
print(train['PitNextLap'].value_counts())

fig, ax = plt.subplots(figsize=(5, 4))
train['PitNextLap'].value_counts().plot(kind='bar', ax=ax, color=['#4472C4', '#ED7D31'])
ax.set_title('PitNextLap class balance')
ax.set_xticklabels(['No Pit (0)', 'Pit (1)'], rotation=0)
for p in ax.patches:
    ax.annotate(f'{p.get_height():,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom')
plt.tight_layout()
plt.show()

## 3. The 2023 Year Anomaly (critical)

In [ ]:
year_stats = train.groupby('Year').agg(
    rows=('PitNextLap', 'count'),
    pit_rate=('PitNextLap', 'mean'),
    pitstop_rate=('PitStop', 'mean')
).round(4)
print(year_stats)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
year_stats['pit_rate'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('PitNextLap rate by Year')
axes[0].set_ylabel('Rate')
axes[0].set_xticklabels(year_stats.index, rotation=0)

year_stats['pitstop_rate'].plot(kind='bar', ax=axes[1], color='darkorange')
axes[1].set_title('PitStop (current lap) rate by Year')
axes[1].set_xticklabels(year_stats.index, rotation=0)
plt.tight_layout()
plt.show()
print('NOTE: 2023 has anomalously low pit rates — synthetic generation artifact. is_year2023 will be the most important feature.')

In [ ]:
# Test set year distribution
print('Test set Year distribution:')
print(test['Year'].value_counts().sort_index())

## 4. Tyre life vs pit decision by compound

In [ ]:
compounds = train['Compound'].dropna().unique()
print('Compounds:', sorted(compounds))
print(train['Compound'].value_counts())

In [ ]:
# Exclude 2023 for cleaner signal
non2023 = train[train['Year'] != 2023]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, compound in enumerate(['SOFT', 'MEDIUM', 'HARD']):
    subset = non2023[non2023['Compound'] == compound]
    subset[subset['PitNextLap'] == 0]['TyreLife'].hist(
        ax=axes[i], bins=40, alpha=0.5, label='No Pit', density=True, color='steelblue')
    subset[subset['PitNextLap'] == 1]['TyreLife'].hist(
        ax=axes[i], bins=40, alpha=0.5, label='Pit Next', density=True, color='darkorange')
    median_pit = subset[subset['PitNextLap'] == 1]['TyreLife'].median()
    axes[i].axvline(median_pit, color='red', linestyle='--', label=f'Median pit TyreLife={median_pit:.0f}')
    axes[i].set_title(f'{compound} compound')
    axes[i].set_xlabel('TyreLife (laps)')
    axes[i].legend(fontsize=8)
plt.suptitle('TyreLife distribution at pit vs no-pit (non-2023)', y=1.02)
plt.tight_layout()
plt.show()

## 5. Race progress and pit timing

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
non2023[non2023['PitNextLap'] == 1]['RaceProgress'].hist(
    bins=50, ax=axes[0], color='darkorange', density=True)
axes[0].set_title('RaceProgress when PitNextLap=1 (non-2023)')
axes[0].set_xlabel('RaceProgress')

rp_bins = pd.cut(non2023['RaceProgress'], bins=20)
pit_by_rp = non2023.groupby(rp_bins)['PitNextLap'].mean()
pit_by_rp.plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Pit rate by RaceProgress bucket')
axes[1].set_xticklabels([str(b) for b in pit_by_rp.index], rotation=45, ha='right', fontsize=7)
plt.tight_layout()
plt.show()

## 6. Cumulative degradation distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
non2023[non2023['PitNextLap'] == 0]['Cumulative_Degradation'].clip(-5, 5).hist(
    bins=80, alpha=0.5, label='No Pit', density=True, color='steelblue', ax=ax)
non2023[non2023['PitNextLap'] == 1]['Cumulative_Degradation'].clip(-5, 5).hist(
    bins=80, alpha=0.5, label='Pit Next', density=True, color='darkorange', ax=ax)
ax.set_title('Cumulative_Degradation distribution (clipped -5 to 5, non-2023)')
ax.legend()
plt.tight_layout()
plt.show()

neg_pct = (train['Cumulative_Degradation'] < 0).mean()
print(f'Fraction of rows with negative Cumulative_Degradation: {neg_pct:.3f}')

## 7. Lap time trends

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

non2023[non2023['PitNextLap'] == 0]['LapTime (s)'].clip(60, 150).hist(
    bins=80, alpha=0.5, label='No Pit', density=True, color='steelblue', ax=axes[0])
non2023[non2023['PitNextLap'] == 1]['LapTime (s)'].clip(60, 150).hist(
    bins=80, alpha=0.5, label='Pit Next', density=True, color='darkorange', ax=axes[0])
axes[0].set_title('LapTime(s) distribution (non-2023)')
axes[0].legend()

non2023[non2023['PitNextLap'] == 0]['LapTime_Delta'].clip(-5, 5).hist(
    bins=80, alpha=0.5, label='No Pit', density=True, color='steelblue', ax=axes[1])
non2023[non2023['PitNextLap'] == 1]['LapTime_Delta'].clip(-5, 5).hist(
    bins=80, alpha=0.5, label='Pit Next', density=True, color='darkorange', ax=axes[1])
axes[1].set_title('LapTime_Delta distribution (non-2023)')
axes[1].legend()
plt.tight_layout()
plt.show()

## 8. Driver encoding analysis

In [ ]:
import re
real_drivers = train[~train['Driver'].astype(str).str.match(r'^D\d+$')]['Driver'].unique()
enc_drivers = train[train['Driver'].astype(str).str.match(r'^D\d+$')]['Driver'].unique()
print(f'Real driver abbreviations: {len(real_drivers)}')
print(f'Encoded driver IDs (Dxxx): {len(enc_drivers)}')
print('Sample real:', sorted(real_drivers)[:10])
print('Sample encoded:', sorted(enc_drivers)[:10])

## 9. External dataset overview

In [ ]:
print(f'External dataset shape (raw): {ext_raw.shape}')
print(f'External dataset shape (after dedup): {ext.shape}')
print(f'Overlap removed: {len(ext_raw) - len(ext):,} rows')
print(f'\nExternal PitNextLap rate: {ext["PitNextLap"].mean():.4f}')
print(f'Train PitNextLap rate:    {train["PitNextLap"].mean():.4f}')
print(f'\nExternal columns not in train: {set(ext.columns) - set(train.columns)}')

## 10. Feature correlations with target

In [ ]:
num_cols = ['TyreLife', 'Stint', 'LapNumber', 'Position', 'RaceProgress',
            'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'Position_Change']

corrs = train[num_cols + ['PitNextLap']].corr()['PitNextLap'].drop('PitNextLap').sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
corrs.plot(kind='barh', ax=ax, color=['red' if c < 0 else 'steelblue' for c in corrs])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Pearson correlation of raw features with PitNextLap')
plt.tight_layout()
plt.show()
print(corrs)

## 11. Race-level pit strategy variation

In [ ]:
race_pit_rate = (non2023.groupby('Race')['PitNextLap'].mean()
                 .sort_values(ascending=True))
fig, ax = plt.subplots(figsize=(12, 6))
race_pit_rate.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('PitNextLap rate by Race (non-2023)')
ax.set_xlabel('Pit rate')
plt.tight_layout()
plt.show()

## Summary

Key findings:
- **2023 anomaly confirmed**: ~1% pit rate vs 25-30% other years → `is_year2023` is critical
- **Class imbalance**: ~20% overall positive rate (masks 2023 anomaly)
- **TyreLife thresholds**: SOFT pits ~12 laps, MEDIUM ~16, HARD ~20 — not sharp cutoffs
- **Race progress**: Pit peak at 0.4–0.7, drops after 0.8
- **External data**: 25.5% pit rate vs 19.9% train — distribution shift to handle
- **Driver encoding**: 2 types (real 3-letter vs Dxxx encoded) → `is_real_driver` flag needed